# DDMRP Classification Model Pipeline

End-to-end ML pipeline for Color_buffer classification from Bronze to trained model, following medallion architecture (Bronze → Silver → Model).

In [0]:
# Install required libraries for ML pipeline
%pip install pandas openpyxl scikit-learn mlflow category_encoders matplotlib seaborn

In [0]:
# Restart kernel after installing libraries
dbutils.library.restartPython()

In [0]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
from mlflow.models import infer_signature

# Set display options
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

In [0]:
# Bronze Layer: Load raw data from Unity Catalog table
import os

# Load data from Unity Catalog table
table_name = "workspace.default.DDMRP"

print("=" * 60)
print("LOADING DATA FROM UNITY CATALOG")
print("=" * 60)
print(f"\n📊 Table: {table_name}")

try:
    # Read from Unity Catalog using Spark
    df_spark = spark.table(table_name)
    
    # Convert to pandas for processing
    df_bronze = df_spark.toPandas()
    
    print(f"\n✓ Bronze data loaded successfully")
    print(f"  Rows: {df_bronze.shape[0]:,}")
    print(f"  Columns: {df_bronze.shape[1]}")
    
    print(f"\n📋 Column names:")
    for i, col in enumerate(df_bronze.columns, 1):
        print(f"  {i:2d}. {col}")
    
    print(f"\n🔍 Data preview (first 5 rows):")
    display(df_bronze.head())
    
    print(f"\n📈 Memory usage: {df_bronze.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
except Exception as e:
    print(f"\n❌ ERROR loading table: {str(e)}")
    print(f"\n💡 Verifique que la tabla '{table_name}' existe")
    print("\nTablas disponibles en workspace.default:")
    try:
        tables = spark.sql("SHOW TABLES IN workspace.default").collect()
        for t in tables:
            print(f"  - {t.tableName}")
    except:
        pass

In [0]:
# Exploratory Data Analysis
print("=== Data Structure ===")
print(f"Shape: {df_bronze.shape}")
print(f"\nData Types:")
for col in df_bronze.columns:
    print(f"{col}: {df_bronze[col].dtype}")

print(f"\nMissing Values:")
missing_df = pd.DataFrame({
    'Column': df_bronze.columns,
    'Missing_Count': df_bronze.isnull().sum().values,
    'Missing_Percent': (df_bronze.isnull().sum().values / len(df_bronze) * 100).round(2)
})
print(missing_df[missing_df['Missing_Count'] > 0])

print(f"\nBasic Statistics (Numeric Columns):")
print(df_bronze.describe())

# Check for target variable (Color_Buffer)
if 'Color_Buffer' in df_bronze.columns:
    print(f"\n=== Target Variable: Color_Buffer ===")
    print(df_bronze['Color_Buffer'].value_counts())
else:
    print("\nTarget column not found")

In [0]:
# Classify feature types
feature_types = []

for col in df_bronze.columns:
    dtype = df_bronze[col].dtype
    unique_count = df_bronze[col].nunique()
    
    if dtype in ['float64', 'float32']:
        feature_type = 'Numeric (Float)'
    elif dtype in ['int64', 'int32']:
        if unique_count < 10:
            feature_type = 'Categorical (Integer)'
        else:
            feature_type = 'Numeric (Integer)'
    elif dtype == 'object':
        if unique_count < 20:
            feature_type = 'Categorical (String)'
        else:
            feature_type = 'Text/String'
    elif dtype == 'bool':
        feature_type = 'Categorical (Boolean)'
    else:
        feature_type = str(dtype)
    
    feature_types.append({'Feature': col, 'Type': feature_type, 'Unique_Values': unique_count})

feature_types_df = pd.DataFrame(feature_types)
print("\n=== Feature Type Classification ===")
display(feature_types_df)

In [0]:
# Visualize target variable distribution
if 'Color_Buffer' in df_bronze.columns:
    plt.figure(figsize=(10, 5))
    
    # Count plot
    plt.subplot(1, 2, 1)
    df_bronze['Color_Buffer'].value_counts().plot(kind='bar')
    plt.title('Color_Buffer Class Distribution')
    plt.xlabel('Class')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    
    # Pie chart
    plt.subplot(1, 2, 2)
    df_bronze['Color_Buffer'].value_counts().plot(kind='pie', autopct='%1.1f%%')
    plt.title('Color_Buffer Class Proportions')
    plt.ylabel('')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nClass balance check:")
    print(df_bronze['Color_Buffer'].value_counts(normalize=True))
else:
    print("Please specify the correct target column name for Color_Buffer classification.")

In [0]:
# Correlation analysis for numeric features
numeric_cols = df_bronze.select_dtypes(include=[np.number]).columns.tolist()

if len(numeric_cols) > 1:
    plt.figure(figsize=(12, 8))
    correlation_matrix = df_bronze[numeric_cols].corr()
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
    plt.title('Feature Correlation Matrix')
    plt.tight_layout()
    plt.show()
    
    # Check for high correlations with target (if numeric)
    if 'Color_Buffer' in df_bronze.columns:
        # If Color_Buffer is categorical, encode it temporarily for correlation
        df_temp = df_bronze.copy()
        if df_temp['Color_Buffer'].dtype == 'object':
            df_temp['Color_Buffer_encoded'] = pd.factorize(df_temp['Color_Buffer'])[0]
            target_col = 'Color_Buffer_encoded'
        else:
            target_col = 'Color_Buffer'
        
        if target_col in df_temp.select_dtypes(include=[np.number]).columns:
            correlations = df_temp[numeric_cols].corrwith(df_temp[target_col]).abs().sort_values(ascending=False)
            print("\n=== Feature Correlations with Target ===")
            print(correlations)
else:
    print("Not enough numeric features for correlation analysis.")

In [0]:
# Silver Layer: Clean data and select features for Color_buffer classification

# First, identify the target column
if 'Color_Buffer' not in df_bronze.columns:
    print("Note: 'Color_Buffer' column not found. Available columns:")
    print(df_bronze.columns.tolist())
    print("\nPlease update this cell with the correct target column name.")
else:
    # Create silver dataframe
    df_silver = df_bronze.copy()
    
    # Remove rows with missing target
    df_silver = df_silver[df_silver['Color_Buffer'].notna()]
    
    # Suppress fields (example: remove ID columns, timestamps, or irrelevant fields)
    # Adjust this list based on your actual data
    fields_to_suppress = []  # Add field names to suppress here
    
    # Identify columns to potentially suppress (example logic)
    for col in df_silver.columns:
        # Suppress high-cardinality text fields (likely IDs)
        if df_silver[col].dtype == 'object' and df_silver[col].nunique() > 0.9 * len(df_silver):
            if col != 'Color_Buffer':
                fields_to_suppress.append(col)
    
    if fields_to_suppress:
        print(f"Suppressing fields: {fields_to_suppress}")
        df_silver = df_silver.drop(columns=fields_to_suppress)
    
    print(f"\nSilver data shape: {df_silver.shape}")
    print(f"Remaining features: {[col for col in df_silver.columns if col != 'Color_Buffer']}")
    
    # Convert datetime to numeric (days since a reference date)
    if 'Fecha_Ultimo_Ingreso' in df_silver.columns:
        reference_date = df_silver['Fecha_Ultimo_Ingreso'].min()
        df_silver['Dias_Desde_Ultimo_Ingreso'] = (df_silver['Fecha_Ultimo_Ingreso'] - reference_date).dt.days
        # Drop the original datetime column
        df_silver = df_silver.drop(columns=['Fecha_Ultimo_Ingreso'])
    
    # Separate features and target
    X = df_silver.drop(columns=['Color_Buffer'])
    y = df_silver['Color_Buffer']
    
    print(f"\nFeatures shape: {X.shape}")
    print(f"Target shape: {y.shape}")
    print(f"\nTarget distribution:")
    print(y.value_counts())

In [0]:
# Target Leakage Detection
if 'Color_Buffer' in df_bronze.columns:
    # Encode target for correlation analysis
    y_encoded = pd.factorize(y)[0]
    
    # Calculate correlations with numeric features
    numeric_features = X.select_dtypes(include=[np.number])
    
    if len(numeric_features.columns) > 0:
        correlations = numeric_features.corrwith(pd.Series(y_encoded, index=X.index)).abs().sort_values(ascending=False)
        
        print("=== Target Leakage Check ===")
        print("\nAbsolute correlations with target (sorted):")
        print(correlations)
        
        # Flag high correlations (potential leakage)
        high_corr_threshold = 0.95
        suspects = correlations[correlations > high_corr_threshold]
        
        if len(suspects) > 0:
            print(f"\n⚠️ WARNING: Features with very high correlation (>{high_corr_threshold}):")
            for feat, corr in suspects.items():
                print(f"  - {feat}: {corr:.3f} (potential target leakage)")
            print("\nThese features should be reviewed before training.")
        else:
            print("\n✓ No obvious target leakage detected.")
    else:
        print("No numeric features available for leakage detection.")

In [0]:
# Split data into training and test sets
if 'Color_Buffer' in df_bronze.columns:
    # Check if there's a time-based column for temporal split
    date_cols = [col for col in X.columns if 'date' in col.lower() or 'time' in col.lower()]
    
    if date_cols:
        print(f"Time-based columns found: {date_cols}")
        print("Consider using temporal split if this is time-series data.")
    
    # Use stratified random split (default for classification)
    # Adjust test_size as needed (default: 20%)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, 
        test_size=0.2, 
        random_state=42,
        stratify=y  # Maintains class distribution in both sets
    )
    
    print("=== Data Split Complete ===")
    print(f"Training set: {X_train.shape[0]} samples")
    print(f"Test set: {X_test.shape[0]} samples")
    print(f"\nTrain set class distribution:")
    print(y_train.value_counts(normalize=True))
    print(f"\nTest set class distribution:")
    print(y_test.value_counts(normalize=True))

In [0]:
# Build preprocessing pipeline
if 'Color_Buffer' in df_bronze.columns:
    # Identify numeric and categorical columns
    numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
    
    print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
    print(f"Categorical features ({len(categorical_features)}): {categorical_features}")
    
    # Numeric preprocessing: impute with mean, then scale
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ])
    
    # Categorical preprocessing: impute with most frequent, then one-hot encode
    from sklearn.preprocessing import OneHotEncoder
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    
    # Combine preprocessing steps
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ],
        remainder='passthrough'  # Keep other columns as-is
    )
    
    print("\n✓ Preprocessing pipeline created")

In [0]:
# Train Random Forest Classifier with MLflow tracking
if 'Color_Buffer' in df_bronze.columns:
    # Create full pipeline: preprocessing + model
    model_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            random_state=42,
            n_jobs=-1
        ))
    ])
    
    # Start MLflow run
    mlflow.set_experiment("/Users/luisgalvis2002@gmail.com/DDMRP_Classification")
    
    with mlflow.start_run() as run:
        # Train the model
        print("Training model...")
        model_pipeline.fit(X_train, y_train)
        
        # Make predictions
        y_pred = model_pipeline.predict(X_test)
        
        # Calculate metrics
        accuracy = accuracy_score(y_test, y_pred)
        
        # Log parameters
        mlflow.log_param("model_type", "RandomForest")
        mlflow.log_param("n_estimators", 100)
        mlflow.log_param("max_depth", 10)
        mlflow.log_param("test_size", 0.2)
        mlflow.log_param("random_state", 42)
        
        # Log metrics
        mlflow.log_metric("accuracy", accuracy)
        
        # Log model with signature using cloudpickle serialization
        # Convert any Decimal columns to float for JSON serialization
        X_train_sample = X_train.head(100).copy()
        for col in X_train_sample.select_dtypes(include=['object']).columns:
            try:
                X_train_sample[col] = X_train_sample[col].astype(float)
            except (ValueError, TypeError):
                pass  # Keep as-is if conversion fails (e.g., for string columns)
        signature = infer_signature(X_train_sample, model_pipeline.predict(X_train_sample))
        # Prepare input example with Decimal conversion
        input_example = X_train.head(3).copy()
        for col in input_example.select_dtypes(include=['object']).columns:
            try:
                input_example[col] = input_example[col].astype(float)
            except (ValueError, TypeError):
                pass
        
        model_info = mlflow.sklearn.log_model(
            model_pipeline,
            artifact_path="model",
            signature=signature,
            input_example=input_example,
            serialization_format='cloudpickle'  # Use cloudpickle to avoid skops trust issues
        )
        
        print("\n=== Training Complete ===")
        print(f"Run ID: {run.info.run_id}")
        print(f"Model URI: {model_info.model_uri}")
        print(f"Accuracy: {accuracy:.4f}")

In [0]:
# Detailed model evaluation
if 'Color_Buffer' in df_bronze.columns:
    print("=== Classification Report ===")
    print(classification_report(y_test, y_pred))
    
    # Confusion Matrix
    plt.figure(figsize=(8, 6))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()
    
    # Feature importance (if available)
    if hasattr(model_pipeline.named_steps['classifier'], 'feature_importances_'):
        # Get feature names after preprocessing
        feature_names = []
        
        # Numeric features
        feature_names.extend(numeric_features)
        
        # One-hot encoded categorical features
        if len(categorical_features) > 0:
            cat_encoder = model_pipeline.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
            feature_names.extend(cat_encoder.get_feature_names_out(categorical_features))
        
        # Get importances
        importances = model_pipeline.named_steps['classifier'].feature_importances_
        feature_importance_df = pd.DataFrame({
            'Feature': feature_names[:len(importances)],
            'Importance': importances
        }).sort_values('Importance', ascending=False).head(15)
        
        # Plot feature importance
        plt.figure(figsize=(10, 6))
        plt.barh(feature_importance_df['Feature'], feature_importance_df['Importance'])
        plt.xlabel('Importance')
        plt.title('Top 15 Feature Importances')
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.show()
        
        print("\n=== Top Feature Importances ===")
        display(feature_importance_df)

In [0]:
# ============================================
# 1. ANÁLISIS DE VALIDACIÓN DEL PRONÓSTICO ADU
# ============================================

print("=" * 60)
print("VALIDACIÓN DEL PRONÓSTICO ADU")
print("=" * 60)

# Análisis de la relación entre ADU y el Color_Buffer resultante
print("\n1. Distribución de ADU por Color_Buffer:")
print("-" * 60)
adu_by_buffer = df_bronze.groupby('Color_Buffer')['ADU'].agg(['mean', 'median', 'std', 'min', 'max'])
print(adu_by_buffer.round(2))

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot
df_bronze.boxplot(column='ADU', by='Color_Buffer', ax=axes[0])
axes[0].set_title('Distribución de ADU por Color de Buffer')
axes[0].set_xlabel('Color Buffer')
axes[0].set_ylabel('ADU (Promedio Diario)')
plt.sca(axes[0])
plt.xticks(rotation=45)

# Violin plot para ver densidad
import seaborn as sns
sns.violinplot(data=df_bronze, x='Color_Buffer', y='ADU', ax=axes[1])
axes[1].set_title('Densidad de ADU por Color de Buffer')
axes[1].set_xlabel('Color Buffer')
axes[1].set_ylabel('ADU (Promedio Diario)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Análisis de coherencia ADU vs Topes
print("\n2. Coherencia entre ADU y Topes (Rojo/Amarillo/Verde):")
print("-" * 60)
df_bronze['ADU_Tope_Rojo_Ratio'] = df_bronze['ADU'] / (df_bronze['Tope_Rojo'] + 0.001)  # Evitar división por 0
df_bronze['ADU_Tope_Amarillo_Ratio'] = df_bronze['ADU'] / (df_bronze['Tope_Amarillo'] + 0.001)
df_bronze['ADU_Tope_Verde_Ratio'] = df_bronze['ADU'] / (df_bronze['Tope_Verde'] + 0.001)

print(f"\nRatio ADU/Tope_Rojo - Media: {df_bronze['ADU_Tope_Rojo_Ratio'].mean():.3f}")
print(f"Ratio ADU/Tope_Amarillo - Media: {df_bronze['ADU_Tope_Amarillo_Ratio'].mean():.3f}")
print(f"Ratio ADU/Tope_Verde - Media: {df_bronze['ADU_Tope_Verde_Ratio'].mean():.3f}")

# Identificar productos con inconsistencias
print("\n3. Productos con posibles inconsistencias en ADU:")
print("-" * 60)

# Productos con ADU muy bajo pero alto inventario
inconsistencias = df_bronze[
    ((df_bronze['ADU'] < 5) & (df_bronze['Total_Inventario_Tienda'] > 100)) |
    ((df_bronze['ADU'] > 15) & (df_bronze['Total_Inventario_Tienda'] < 10))
]

print(f"\nProductos con posibles inconsistencias ADU vs Inventario: {len(inconsistencias)}")
print(f"Porcentaje del total: {(len(inconsistencias)/len(df_bronze)*100):.2f}%")

if len(inconsistencias) > 0:
    print("\nMuestra de productos con inconsistencias:")
    display(inconsistencias[['SKU', 'ADU', 'Total_Inventario_Tienda', 'Color_Buffer', 'Dias_Inventario']].head(10))

# Oportunidades de mejora
print("\n" + "="*60)
print("OPORTUNIDADES DE MEJORA EN PRONÓSTICO ADU:")
print("="*60)

oportunidades = []

# Oportunidad 1: Productos con alta variabilidad
high_var_skus = df_bronze.groupby('SKU')['ADU'].std().sort_values(ascending=False).head(10)
if high_var_skus.mean() > 5:
    oportunidades.append("1. Alta variabilidad en ADU por SKU - considerar suavizado estacional")

# Oportunidad 2: Desalineación entre ADU y días de inventario
if df_bronze['Dias_Inventario'].mean() > 5:
    oportunidades.append("2. Días de inventario promedio alto - el ADU podría estar subestimado")

# Oportunidad 3: Productos en sobre-stock
sobre_stock_pct = (df_bronze['Sobre_Stock_Unid'] > 0).sum() / len(df_bronze) * 100
if sobre_stock_pct > 10:
    oportunidades.append(f"3. {sobre_stock_pct:.1f}% productos en sobre-stock - revisar precisión del pronóstico")

for i, oportunidad in enumerate(oportunidades, 1):
    print(f"\n✓ {oportunidad}")

if not oportunidades:
    print("\n✓ El pronóstico ADU parece estar bien estructurado")

In [0]:
# ============================================
# 2. SISTEMA DE PREDICCIÓN PARA PRODUCTOS NUEVOS
# ============================================

import ipywidgets as widgets
from IPython.display import display, clear_output

print("=" * 60)
print("PREDICTOR DE COLOR BUFFER PARA PRODUCTOS NUEVOS")
print("=" * 60)

def predecir_producto_nuevo():
    """Función interactiva para predecir el color buffer de un producto nuevo"""
    
    print("\nIngrese los datos del producto nuevo:")
    print("-" * 60)
    
    # Crear inputs para las características más importantes
    # Basado en las variables que tenemos
    
    producto_nuevo = {
        'ID_Tienda': 'T-01',
        'SKU': 'SKU-NEW-0001',
        'Descripcion_Producto': 'Producto Nuevo',
        'ADU': 10,
        'LT_Dias': 3,
        'Tope_Rojo': 9,
        'Tope_Amarillo': 42,
        'Tope_Verde': 59,
        'Inventario_Bodega': 500,
        'Inventario_Tienda': 30,
        'Stock_Transito': 10,
        'Total_Inventario_Tienda': 40,
        'Unidades_a_Enviar': 0,
        'Sobre_Stock_Unid': 0,
        'Stock_Vitrina': 8,
        'Costo_Unitario': 200000,
        'Costo_Inventario': 8000000,
        'Costo_Sobre_Stock': 0,
        'Dias_Inventario': 3.0,
        'Dias_Desde_Ultimo_Ingreso': 30
    }
    
    return producto_nuevo

# Función para hacer predicciones
def hacer_prediccion(datos_producto):
    """Realiza la predicción del color buffer"""
    
    # Convertir a DataFrame
    df_nuevo = pd.DataFrame([datos_producto])
    
    # Hacer predicción
    prediccion = model_pipeline.predict(df_nuevo)[0]
    
    # Obtener probabilidades
    probabilidades = model_pipeline.predict_proba(df_nuevo)[0]
    clases = model_pipeline.classes_
    
    # Crear DataFrame de probabilidades
    prob_df = pd.DataFrame({
        'Color_Buffer': clases,
        'Probabilidad': probabilidades,
        'Porcentaje': probabilidades * 100
    }).sort_values('Probabilidad', ascending=False)
    
    return prediccion, prob_df

# Ejemplo de predicción
print("\nEjemplo: Producto con valores promedio")
producto_ejemplo = predecir_producto_nuevo()
prediccion, probabilidades = hacer_prediccion(producto_ejemplo)

print(f"\n{'='*60}")
print(f"PREDICCIÓN: El producto quedará en color {prediccion}")
print(f"{'='*60}")

print("\nProbabilidades por color:")
print("-" * 60)
display(probabilidades)

# Visualización de probabilidades
fig, ax = plt.subplots(figsize=(10, 6))
colors_map = {'ROJO': 'red', 'AMARILLO': 'yellow', 'VERDE': 'green', 'AZUL': 'blue', 'NEGRO': 'black'}
bar_colors = [colors_map.get(c, 'gray') for c in probabilidades['Color_Buffer']]

ax.barh(probabilidades['Color_Buffer'], probabilidades['Porcentaje'], color=bar_colors, alpha=0.7, edgecolor='black')
ax.set_xlabel('Probabilidad (%)', fontsize=12)
ax.set_title('Probabilidad de Color Buffer para Producto Nuevo', fontsize=14, fontweight='bold')
ax.invert_yaxis()

for i, (color, prob) in enumerate(zip(probabilidades['Color_Buffer'], probabilidades['Porcentaje'])):
    ax.text(prob + 1, i, f'{prob:.1f}%', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("Para predecir otro producto, modifique los valores en la")
print("función predecir_producto_nuevo() y ejecute nuevamente.")
print("="*60)

## Conversión de Excel a Parquet

Esta sección convierte el archivo Excel original a formato Parquet para:
* **Mejor rendimiento**: Lectura/escritura más rápida
* **Compresión eficiente**: Menor uso de almacenamiento
* **Compatibilidad Spark**: Mejor integración con Databricks
* **Soporte columnar**: Optimizado para análisis

In [0]:
# ============================================
# CONVERSIÓN DE EXCEL A PARQUET
# ============================================

import pandas as pd
import os

print("=" * 60)
print("CONVERSIÓN DE DATOS DDMRP: EXCEL → PARQUET")
print("=" * 60)

# Rutas de archivos - using UC table instead of direct file access
uc_table = "workspace.default.DDMRP"
parquet_path = "/Volumes/workspace/default/my_volume/DDMRP.parquet"

print(f"\n📂 Cargando datos desde Unity Catalog...")
print(f"   Origen: {uc_table}")

# Cargar desde UC table y escribir directamente con Spark
df_spark = spark.table(uc_table)

print(f"\n✓ Datos cargados exitosamente")
print(f"   Filas: {df_spark.count():,}")
print(f"   Columnas: {len(df_spark.columns)}")

# Guardar como Parquet con compresión usando Spark (evita problemas de serialización)
print(f"\n💾 Guardando en formato Parquet...")
print(f"   Destino: {parquet_path}")

df_spark.coalesce(1).write.mode('overwrite').parquet(parquet_path)

print(f"\n✅ Conversión completada exitosamente")
print(f"\n📊 Información del archivo:")
print(f"   Formato: Parquet")
print(f"   Compresión: Snappy")
print(f"   Engine: PyArrow")

In [0]:
# Verificar que el archivo Parquet se creó correctamente
import os

print("=" * 60)
print("VERIFICACIÓN Y COMPARACIÓN")
print("=" * 60)

# Cargar Parquet para verificar
df_parquet = pd.read_parquet(parquet_path)

print(f"\n✓ Archivo Parquet verificado")
print(f"   Filas: {df_parquet.shape[0]:,}")
print(f"   Columnas: {df_parquet.shape[1]}")

# Comparar con datos originales
if df.shape == df_parquet.shape:
    print(f"\n✓ Las dimensiones coinciden perfectamente")
else:
    print(f"\n⚠️ Advertencia: Las dimensiones no coinciden")

# Verificar que los datos son idénticos
print(f"\n🔍 Verificando integridad de datos...")

# Comparar columnas
if set(df.columns) == set(df_parquet.columns):
    print(f"   ✓ Todas las columnas están presentes")
else:
    print(f"   ⚠️ Diferencias en columnas detectadas")

# Mostrar primeras filas
print(f"\n📋 Primeras filas del archivo Parquet:")
display(df_parquet.head())

print(f"\n✅ Archivo Parquet listo para usar en: {parquet_path}")

In [0]:
# ============================================
# CONVERSIÓN USANDO SPARK (ALTERNATIVA)
# ============================================
# Esta celda ofrece una alternativa usando Spark
# para archivos muy grandes o cuando se necesita
# particionamiento

print("=" * 60)
print("CONVERSIÓN USANDO SPARK")
print("=" * 60)

# Rutas - using UC table instead of Excel file
uc_table = "workspace.default.DDMRP"
parquet_path_spark = "/Volumes/workspace/default/my_volume/DDMRP_spark.parquet"

print(f"\n📂 Cargando desde Unity Catalog...")

# Cargar directamente desde UC table (más eficiente que Excel)
df_spark = spark.table(uc_table)

print(f"\n✓ DataFrame Spark creado")

# Mostrar esquema
print(f"\n📋 Esquema del DataFrame:")
df_spark.printSchema()

# Guardar como Parquet
print(f"\n💾 Guardando como Parquet con Spark...")
df_spark.write.mode('overwrite').parquet(parquet_path_spark)

print(f"\n✅ Archivo Parquet (Spark) guardado en: {parquet_path_spark}")

# Leer y verificar
df_spark_read = spark.read.parquet(parquet_path_spark)
print(f"\n🔍 Verificación:")
print(f"   Filas: {df_spark_read.count():,}")
print(f"   Columnas: {len(df_spark_read.columns)}")

## Cómo usar el archivo Parquet en el pipeline

**Opción 1: Con Pandas**
```python
df_bronze = pd.read_parquet("/Volumes/workspace/bronze/ddmrp_bronze/DDMRP.parquet")
```

**Opción 2: Con Spark**
```python
df_spark = spark.read.parquet("/Volumes/workspace/bronze/ddmrp_bronze/DDMRP.parquet")
df_bronze = df_spark.toPandas()  # Convertir a Pandas si es necesario
```

### Ventajas del formato Parquet:
* ⚡ **10-100x más rápido** para leer datos
* 💾 **50-90% menos espacio** en disco
* 🔄 **Compatible con Spark** nativamente
* 📊 **Preserva tipos de datos** automáticamente
* 🎯 **Lectura columnar** eficiente

In [0]:
# ============================================
# 3. ANÁLISIS TEMPORAL - ANTES Y DESPUÉS
# ============================================

print("=" * 60)
print("ANÁLISIS TEMPORAL: ANTES Y DESPUÉS")
print("=" * 60)

# Dividir los datos por la mediana de la fecha
mediana_fecha = df_bronze['Fecha_Ultimo_Ingreso'].median()

df_antes = df_bronze[df_bronze['Fecha_Ultimo_Ingreso'] <= mediana_fecha].copy()
df_despues = df_bronze[df_bronze['Fecha_Ultimo_Ingreso'] > mediana_fecha].copy()

print(f"\nFecha de corte (mediana): {mediana_fecha.date()}")
print(f"\nPeriodo ANTES: {df_antes['Fecha_Ultimo_Ingreso'].min().date()} a {df_antes['Fecha_Ultimo_Ingreso'].max().date()}")
print(f"Registros ANTES: {len(df_antes):,}")

print(f"\nPeriodo DESPUÉS: {df_despues['Fecha_Ultimo_Ingreso'].min().date()} a {df_despues['Fecha_Ultimo_Ingreso'].max().date()}")
print(f"Registros DESPUÉS: {len(df_despues):,}")

# Comparación de distribución de Color_Buffer
print("\n" + "="*60)
print("COMPARACIÓN DE DISTRIBUCIÓN DE COLOR BUFFER")
print("="*60)

comparacion = pd.DataFrame({
    'ANTES_Count': df_antes['Color_Buffer'].value_counts(),
    'ANTES_%': df_antes['Color_Buffer'].value_counts(normalize=True) * 100,
    'DESPUÉS_Count': df_despues['Color_Buffer'].value_counts(),
    'DESPUÉS_%': df_despues['Color_Buffer'].value_counts(normalize=True) * 100
})
comparacion['Cambio_%'] = comparacion['DESPUÉS_%'] - comparacion['ANTES_%']

print("\nDistribución de Color Buffer - Antes vs Después:")
display(comparacion.round(2))

# Visualización comparativa
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ANTES
df_antes['Color_Buffer'].value_counts().plot(kind='bar', ax=axes[0], color=['yellow', 'green', 'blue', 'red', 'black'], alpha=0.7)
axes[0].set_title('ANTES - Distribución de Color Buffer', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Color Buffer')
axes[0].set_ylabel('Cantidad')
axes[0].tick_params(axis='x', rotation=45)

# DESPUÉS
df_despues['Color_Buffer'].value_counts().plot(kind='bar', ax=axes[1], color=['yellow', 'green', 'blue', 'red', 'black'], alpha=0.7)
axes[1].set_title('DESPUÉS - Distribución de Color Buffer', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Color Buffer')
axes[1].set_ylabel('Cantidad')
axes[1].tick_params(axis='x', rotation=45)

# CAMBIO
comparacion['Cambio_%'].plot(kind='bar', ax=axes[2], color=['red' if x < 0 else 'green' for x in comparacion['Cambio_%']])
axes[2].set_title('CAMBIO en % - Antes vs Después', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Color Buffer')
axes[2].set_ylabel('Cambio en Puntos Porcentuales')
axes[2].axhline(y=0, color='black', linestyle='--', linewidth=1)
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Comparación de métricas clave
print("\n" + "="*60)
print("MÉTRICAS CLAVE - COMPARACIÓN TEMPORAL")
print("="*60)

metricas_antes = pd.DataFrame({
    'ADU': [df_antes['ADU'].mean()],
    'Dias_Inventario': [df_antes['Dias_Inventario'].mean()],
    'Sobre_Stock_%': [(df_antes['Sobre_Stock_Unid'] > 0).sum() / len(df_antes) * 100],
    'Unidades_Enviar': [df_antes['Unidades_a_Enviar'].mean()],
    'Inventario_Total': [df_antes['Total_Inventario_Tienda'].mean()]
}, index=['ANTES'])

metricas_despues = pd.DataFrame({
    'ADU': [df_despues['ADU'].mean()],
    'Dias_Inventario': [df_despues['Dias_Inventario'].mean()],
    'Sobre_Stock_%': [(df_despues['Sobre_Stock_Unid'] > 0).sum() / len(df_despues) * 100],
    'Unidades_Enviar': [df_despues['Unidades_a_Enviar'].mean()],
    'Inventario_Total': [df_despues['Total_Inventario_Tienda'].mean()]
}, index=['DESPUÉS'])

metricas_comparacion = pd.concat([metricas_antes, metricas_despues])
metricas_comparacion.loc['DIFERENCIA'] = metricas_comparacion.loc['DESPUÉS'] - metricas_comparacion.loc['ANTES']
metricas_comparacion.loc['CAMBIO_%'] = (metricas_comparacion.loc['DESPUÉS'] / metricas_comparacion.loc['ANTES'] - 1) * 100

print("\nComparación de Métricas Clave:")
display(metricas_comparacion.round(2))

In [0]:
# ============================================
# 4. TOP 6 VARIABLES MÁS DETERMINANTES
# ============================================

print("=" * 60)
print("TOP 6 VARIABLES MÁS DETERMINANTES PARA COLOR BUFFER")
print("=" * 60)

# Obtener importancias de todas las features
feature_names = []
feature_names.extend(numeric_features)

if len(categorical_features) > 0:
    cat_encoder = model_pipeline.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
    feature_names.extend(cat_encoder.get_feature_names_out(categorical_features))

importances = model_pipeline.named_steps['classifier'].feature_importances_

# Crear DataFrame completo de importancias
all_feature_importance = pd.DataFrame({
    'Variable': feature_names[:len(importances)],
    'Importancia': importances,
    'Importancia_%': importances * 100
}).sort_values('Importancia', ascending=False)

# Top 6 variables
top_6 = all_feature_importance.head(6).copy()
top_6['Ranking'] = range(1, 7)

print("\nLas 6 variables más determinantes:")
print("-" * 60)
for idx, row in top_6.iterrows():
    print(f"{row['Ranking']}. {row['Variable']:<30} {row['Importancia_%']:>6.2f}%")

print("\n" + "="*60)

# Visualización detallada
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Gráfico 1: Barras horizontales del Top 6
ax1 = axes[0, 0]
colors = plt.cm.viridis(np.linspace(0.3, 0.9, 6))
ax1.barh(top_6['Variable'], top_6['Importancia_%'], color=colors, edgecolor='black', linewidth=1.5)
ax1.set_xlabel('Importancia (%)', fontsize=12, fontweight='bold')
ax1.set_title('Top 6 Variables Determinantes', fontsize=14, fontweight='bold')
ax1.invert_yaxis()
for i, (var, imp) in enumerate(zip(top_6['Variable'], top_6['Importancia_%'])):
    ax1.text(imp + 0.5, i, f'{imp:.2f}%', va='center', fontweight='bold')

# Gráfico 2: Pie chart del Top 6 vs Resto
ax2 = axes[0, 1]
top_6_total = top_6['Importancia_%'].sum()
resto = 100 - top_6_total
ax2.pie([top_6_total, resto], labels=['Top 6 Variables', 'Otras Variables'], 
        autopct='%1.1f%%', startangle=90, colors=['#ff9999', '#66b3ff'])
ax2.set_title('Contribución del Top 6 vs Resto', fontsize=14, fontweight='bold')

# Gráfico 3: Contribución acumulada
ax3 = axes[1, 0]
cumulative = top_6['Importancia_%'].cumsum()
ax3.plot(range(1, 7), cumulative, marker='o', linewidth=2, markersize=10, color='darkblue')
ax3.fill_between(range(1, 7), cumulative, alpha=0.3, color='blue')
ax3.set_xlabel('Número de Variables', fontsize=12, fontweight='bold')
ax3.set_ylabel('Importancia Acumulada (%)', fontsize=12, fontweight='bold')
ax3.set_title('Importancia Acumulada - Top 6', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.set_xticks(range(1, 7))
for i, val in enumerate(cumulative, 1):
    ax3.text(i, val + 1, f'{val:.1f}%', ha='center', fontweight='bold')

# Gráfico 4: Comparación relativa
ax4 = axes[1, 1]
top_6_sorted = top_6.sort_values('Importancia_%', ascending=True)
colors_relative = ['gold' if i == len(top_6_sorted)-1 else 'skyblue' for i in range(len(top_6_sorted))]
ax4.barh(range(len(top_6_sorted)), top_6_sorted['Importancia_%'], color=colors_relative, edgecolor='black', linewidth=1.5)
ax4.set_yticks(range(len(top_6_sorted)))
ax4.set_yticklabels(top_6_sorted['Variable'])
ax4.set_xlabel('Importancia (%)', fontsize=12, fontweight='bold')
ax4.set_title('Ranking de Importancia (La más alta en dorado)', fontsize=14, fontweight='bold')
for i, val in enumerate(top_6_sorted['Importancia_%']):
    ax4.text(val + 0.5, i, f'{val:.2f}%', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

# Identificar la variable más determinante
variable_mas_importante = top_6.iloc[0]

print("\n" + "="*60)
print("VARIABLE MÁS DETERMINANTE (LA QUE MÁS LE PEGA)")
print("="*60)
print(f"\n🏆 {variable_mas_importante['Variable']}")
print(f"\n   Importancia: {variable_mas_importante['Importancia_%']:.2f}%")
print(f"   Ranking: #{variable_mas_importante['Ranking']}")
print(f"\n   Esta variable tiene {variable_mas_importante['Importancia_%'] / top_6.iloc[1]['Importancia_%']:.2f}x más impacto")
print(f"   que la segunda variable más importante ({top_6.iloc[1]['Variable']})")
print("\n" + "="*60)

In [0]:
# ============================================
# 5. CAPA GOLD - SOLO VARIABLES NECESARIAS
# ============================================

print("=" * 60)
print("CAPA GOLD - DATOS OPTIMIZADOS")
print("=" * 60)

# Seleccionar solo las Top 6 variables + algunas claves para identificación
top_6_variables = top_6['Variable'].tolist()

print("\nVariables seleccionadas para la Capa Gold:")
print("-" * 60)
print("\nVariables de Identificación:")
print("  - SKU")
print("  - Color_Buffer (Target)")

print("\nTop 6 Variables Predictivas:")
for i, var in enumerate(top_6_variables, 1):
    print(f"  {i}. {var}")

# Crear dataset Gold
variables_gold = ['SKU', 'Color_Buffer'] + top_6_variables

# Verificar qué variables existen en el dataframe original
variables_disponibles = [v for v in variables_gold if v in df_bronze.columns]

if len(variables_disponibles) < len(variables_gold):
    print("\n⚠️ Nota: Algunas variables del Top 6 son derivadas (one-hot encoding).")
    print("   Usando las variables originales correspondientes...\n")
    
    # Mapear variables one-hot a sus originales
    variables_base = ['SKU', 'Color_Buffer']
    for var in top_6_variables:
        # Extraer nombre base si es one-hot encoded
        if '_' in var and any(cat in var for cat in ['ID_Tienda', 'SKU', 'Descripcion']):
            base_var = var.split('_')[0] + '_' + var.split('_')[1] if var.count('_') > 1 else var.split('_')[0]
            if base_var in df_bronze.columns and base_var not in variables_base:
                variables_base.append(base_var)
        elif var in df_bronze.columns:
            variables_base.append(var)
    
    variables_gold = list(set(variables_base))

df_gold = df_bronze[variables_gold].copy()

print(f"\nCapa Gold creada: {df_gold.shape[0]:,} registros x {df_gold.shape[1]} variables")
print("\n" + "="*60)

# Mostrar muestra
print("\nMuestra de la Capa Gold:")
display(df_gold.head(10))

# Estadísticas de la capa Gold
print("\n" + "="*60)
print("ESTADÍSTICAS DE LA CAPA GOLD")
print("="*60)

print("\nVariables numéricas en Gold:")
vars_numericas_gold = df_gold.select_dtypes(include=[np.number]).columns.tolist()
if vars_numericas_gold:
    display(df_gold[vars_numericas_gold].describe().round(2))

print("\nDistribución del Target en Gold:")
print(df_gold['Color_Buffer'].value_counts())

# Guardar resumen
print("\n" + "="*60)
print("RESUMEN GOLD LAYER")
print("="*60)
print(f"\n✓ Reducción de variables: {len(df_bronze.columns)} → {len(df_gold.columns)}")
print(f"✓ Reducción de dimensionalidad: {(1 - len(df_gold.columns)/len(df_bronze.columns))*100:.1f}%")
print(f"✓ Registros mantenidos: {len(df_gold):,} (100%)")
print(f"\n✓ Variables Gold optimizadas para:")
print("  - Predicción más rápida")
print("  - Menor almacenamiento")
print("  - Mejor interpretabilidad")
print("  - Enfoque en variables críticas")

In [0]:
# ============================================
# 6. MODELO OPTIMIZADO CON VARIABLES GOLD
# ============================================

print("=" * 60)
print("ENTRENAMIENTO DE MODELO OPTIMIZADO (GOLD)")
print("=" * 60)

# Preparar datos Gold para modelado
X_gold = df_gold.drop(columns=['SKU', 'Color_Buffer'])
y_gold = df_gold['Color_Buffer']

print(f"\nFeatures Gold: {X_gold.shape[1]} variables")
print(f"Target: {y_gold.name}")

# Split
X_train_gold, X_test_gold, y_train_gold, y_test_gold = train_test_split(
    X_gold, y_gold, test_size=0.2, random_state=42, stratify=y_gold
)

print(f"\nTrain: {X_train_gold.shape[0]:,} | Test: {X_test_gold.shape[0]:,}")

# Pipeline simplificado (solo variables numéricas en Gold)
numeric_features_gold = X_train_gold.select_dtypes(include=[np.number]).columns.tolist()
categorical_features_gold = X_train_gold.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"\nNuméricas: {len(numeric_features_gold)}")
print(f"Categóricas: {len(categorical_features_gold)}")

# Preprocessor
if len(categorical_features_gold) > 0:
    from sklearn.preprocessing import OneHotEncoder
    preprocessor_gold = ColumnTransformer(
        transformers=[
            ('num', Pipeline([('imputer', SimpleImputer(strategy='mean')), 
                              ('scaler', StandardScaler())]), numeric_features_gold),
            ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                              ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), 
             categorical_features_gold)
        ]
    )
else:
    preprocessor_gold = ColumnTransformer(
        transformers=[
            ('num', Pipeline([('imputer', SimpleImputer(strategy='mean')), 
                              ('scaler', StandardScaler())]), numeric_features_gold)
        ]
    )

# Modelo Gold
model_gold = Pipeline([
    ('preprocessor', preprocessor_gold),
    ('classifier', RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1))
])

# Entrenar
print("\nEntrenando modelo Gold...")
model_gold.fit(X_train_gold, y_train_gold)

# Evaluar
y_pred_gold = model_gold.predict(X_test_gold)
accuracy_gold = accuracy_score(y_test_gold, y_pred_gold)

print("\n" + "="*60)
print("RESULTADOS MODELO GOLD")
print("="*60)
print(f"\nAccuracy: {accuracy_gold:.4f}")

print("\nComparación con Modelo Completo:")
print("-" * 60)
print(f"Modelo Completo (20 variables): {accuracy:.4f}")
print(f"Modelo Gold ({X_gold.shape[1]} variables):   {accuracy_gold:.4f}")
print(f"\nDiferencia: {abs(accuracy - accuracy_gold):.4f}")

if accuracy_gold >= accuracy - 0.02:
    print("\n✓ El modelo Gold mantiene performance similar con menos variables!")
else:
    print("\n⚠️ El modelo Gold tiene menor accuracy, pero es más interpretable.")

print("\n" + "="*60)

In [0]:
# ============================================
# 7. RESUMEN EJECUTIVO Y RECOMENDACIONES
# ============================================

print("\n\n")
print("#" * 80)
print("#" + " " * 78 + "#")
print("#" + " " * 20 + "RESUMEN EJECUTIVO - ANÁLISIS DDMRP" + " " * 20 + "#")
print("#" + " " * 78 + "#")
print("#" * 80)

print("\n" + "="*80)
print("1. VALIDACIÓN DEL PRONÓSTICO ADU")
print("="*80)
print(f"\n   ADU Promedio: {df_bronze['ADU'].mean():.2f} unidades/día")
print(f"   Días de Inventario Promedio: {df_bronze['Dias_Inventario'].mean():.2f} días")
print(f"   Productos en Sobre-Stock: {(df_bronze['Sobre_Stock_Unid'] > 0).sum():,} ({(df_bronze['Sobre_Stock_Unid'] > 0).sum()/len(df_bronze)*100:.1f}%)")
print("\n   Conclusión: ", end="")
if len(oportunidades) > 0:
    print(f"Se identificaron {len(oportunidades)} oportunidades de mejora")
else:
    print("El pronóstico ADU está bien estructurado")

print("\n" + "="*80)
print("2. TOP 6 VARIABLES MÁS DETERMINANTES PARA COLOR BUFFER")
print("="*80)
for idx, row in top_6.iterrows():
    print(f"   {row['Ranking']}. {row['Variable']:<35} {row['Importancia_%']:>6.2f}%")
    
print(f"\n   Importancia Acumulada del Top 6: {top_6['Importancia_%'].sum():.2f}%")

print("\n" + "="*80)
print("3. VARIABLE QUE MÁS IMPACTA (LA QUE MÁS LE PEGA)")
print("="*80)
print(f"\n   🏆 {variable_mas_importante['Variable']}")
print(f"\n   Esta variable explica el {variable_mas_importante['Importancia_%']:.2f}% de la")
print("   clasificación del Color Buffer.")
print(f"\n   Es {variable_mas_importante['Importancia_%'] / top_6.iloc[1]['Importancia_%']:.2f}x más importante que la segunda variable.")

print("\n" + "="*80)
print("4. ANÁLISIS TEMPORAL (ANTES vs DESPUÉS)")
print("="*80)
print(f"\n   Fecha de Corte: {mediana_fecha.date()}")
print(f"   Registros Antes: {len(df_antes):,}")
print(f"   Registros Después: {len(df_despues):,}")
print("\n   Cambios Principales:")
for color in comparacion.index:
    cambio = comparacion.loc[color, 'Cambio_%']
    if abs(cambio) > 1:
        direccion = "↑" if cambio > 0 else "↓"
        print(f"   {direccion} {color}: {abs(cambio):.1f}% {'incremento' if cambio > 0 else 'decremento'}")

print("\n" + "="*80)
print("5. CAPA GOLD - OPTIMIZACIÓN")
print("="*80)
print(f"\n   Variables originales: {len(df_bronze.columns)}")
print(f"   Variables Gold: {len(df_gold.columns)}")
print(f"   Reducción: {(1 - len(df_gold.columns)/len(df_bronze.columns))*100:.1f}%")
print(f"\n   Accuracy Modelo Completo: {accuracy:.4f}")
print(f"   Accuracy Modelo Gold: {accuracy_gold:.4f}")
print(f"   Diferencia: {abs(accuracy - accuracy_gold):.4f}")

print("\n" + "="*80)
print("6. PREDICCIÓN PARA PRODUCTOS NUEVOS")
print("="*80)
print(f"\n   Sistema de predicción: ✓ Implementado")
print(f"   Ejemplo predicción: {prediccion}")
print(f"   Confianza: {probabilidades.iloc[0]['Porcentaje']:.1f}%")

print("\n" + "="*80)
print("7. RECOMENDACIONES")
print("="*80)
print("\n   ✓ Usar Modelo Gold para predicciones en producción (más rápido, similar accuracy)")
print("   ✓ Monitorear especialmente las Top 6 variables identificadas")
print(f"   ✓ Enfocar esfuerzos de mejora en: {variable_mas_importante['Variable']}")
print("   ✓ Revisar productos con inconsistencias ADU vs Inventario")
print("   ✓ Implementar sistema de predicción para nuevos productos")
if len(oportunidades) > 0:
    print(f"   ✓ Atender las {len(oportunidades)} oportunidades identificadas en el pronóstico ADU")

print("\n" + "#" * 80)
print("\n")